In [ ]:
from datasets import DatasetDict, Dataset
import pandas as pd

base_url = "https://huggingface.co/datasets/HausaNLP/NaijaSenti-Twitter/resolve/refs%2Fconvert%2Fparquet/pcm"

train_df = pd.read_parquet(f"{base_url}/train/0000.parquet")
dev_df = pd.read_parquet(f"{base_url}/validation/0000.parquet")
test_df = pd.read_parquet(f"{base_url}/test/0000.parquet")

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(dev_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)

print(dataset)

print(dataset["train"].features)
print(dataset["train"].unique("label"))

if hasattr(dataset["train"].features["label"], "names"):
    print("Class names:", dataset["train"].features["label"].names)

print("First 10 labels:", dataset["train"]["label"][:10])

print("\n" + "="*60 + "\nFIRST 10 RAW SAMPLES:\n" + "="*60)
for i in range(10):
    sample = dataset["train"][i]
    print(f"[{i+1}] Label: {sample['label']}")
    print(f"    Text:  {sample['tweet']}\n" + "-"*50)
train_texts = set(dataset["train"]["tweet"])
validation_texts = set(dataset["validation"]["tweet"])
test_texts = set(dataset["test"]["tweet"])

print("\nCross-split overlap:")
print("Train ∩ Validation:", len(train_texts & validation_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(validation_texts & test_texts))

df_train = pd.DataFrame(dataset["train"].to_dict())
for label in sorted(df_train["label"].unique()):
    print(f"\nLABEL {label}")
    print(df_train[df_train["label"] == label]["tweet"].head(10).to_string(index=False))

for split in dataset:
    print(f"\n{str(split).upper()}")
    df = pd.DataFrame(dataset[split].to_dict())
    duplicates = df[df["tweet"].duplicated(keep=False)].sort_values("tweet")

    conflicts = (
        duplicates.groupby("tweet")["label"]
        .nunique()
    )

    conflicts = conflicts[conflicts > 1]

    print(f"\n{str(split).upper()}")
    print("Duplicate tweets:", duplicates["tweet"].nunique())
    print("Duplicate tweets with conflicting labels:", len(conflicts))

    if len(conflicts):
        print(
            duplicates[
                duplicates["tweet"].isin(conflicts.index)
            ].head(20)
        )
    print("Rows:", len(df))
    print("Duplicate Tweets:", df["tweet"].duplicated().sum())
    print(df.head())
    print("\nLabel Distribution:")
    print(df["label"].value_counts())
    print(df.isnull().sum())

    df["text_length"] = df["tweet"].astype(str).apply(len)
    print("\nTweet Length Statistics:")
    print(df["text_length"].describe()) 

In [5]:
import os
import re
import html
import pandas as pd
from sklearn.model_selection import train_test_split

base_url = "https://huggingface.co/datasets/HausaNLP/NaijaSenti-Twitter/resolve/refs%2Fconvert%2Fparquet/pcm"

df_raw_train = pd.read_parquet(f"{base_url}/train/0000.parquet")
df_raw_val = pd.read_parquet(f"{base_url}/validation/0000.parquet")
df_raw_test = pd.read_parquet(f"{base_url}/test/0000.parquet")

df_raw_all = pd.concat([df_raw_train, df_raw_val, df_raw_test], ignore_index=True)
os.makedirs("../data/raw", exist_ok=True)
df_raw_all.to_csv("../data/raw/pcm_raw.csv", index=True)
print(f"Raw dataset saved to data/raw folder, {len(df_raw_all)} rows")


def normarlized_text(text: str):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df_raw_all["clean_tweet"] = df_raw_all["tweet"].apply(normarlized_text)

df_dedup = df_raw_all.drop_duplicates(subset=["clean_tweet"], keep="first").copy()
df_dedup = df_dedup.drop(columns=["clean_tweet"])

train_df_, temp_df_ = train_test_split(
    df_dedup, test_size=0.30, random_state=45, stratify=df_dedup["label"]
)
val_df_, test_df_ = train_test_split(
    temp_df_, test_size=0.50, random_state=45, stratify=temp_df_["label"]
)

os.makedirs("../data/processed", exist_ok=True)
train_df_.to_csv("../data/processed/train.csv", index=False)
val_df_.to_csv("../data/processed/val.csv", index=False)
test_df_.to_csv("../data/processed/test.csv", index=False)
print("sanitized, split dataset saved to data/processed folder")

Raw dataset saved to data/raw folder, 10556 rows
sanitized, split dataset saved to data/processed folder
